In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Tokenization isn't always predictable, unbelievably."
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Text:  ", text)
print("Tokens:", tokens)
print("IDs:   ", encoded["input_ids"])
print("Mask:  ", encoded["attention_mask"])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Text:   Tokenization isn't always predictable, unbelievably.
Tokens: ['[CLS]', 'token', '##ization', 'isn', "'", 't', 'always', 'predictable', ',', 'un', '##bel', '##ie', '##va', '##bly', '.', '[SEP]']
IDs:    [101, 19204, 3989, 3475, 1005, 1056, 2467, 21425, 1010, 4895, 8671, 2666, 3567, 6321, 1012, 102]
Mask:   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Rofaida is a computer science learner."
encoded = tokenizer(text)
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
print("Text:  ", text)
print("Tokens:", tokens)
print("IDs:   ", encoded["input_ids"])
print("Mask:  ", encoded["attention_mask"])

Text:   Rofaida is a computer science learner.
Tokens: ['[CLS]', 'ro', '##fa', '##ida', 'is', 'a', 'computer', 'science', 'learn', '##er', '.', '[SEP]']
IDs:    [101, 20996, 7011, 8524, 2003, 1037, 3274, 2671, 4553, 2121, 1012, 102]
Mask:   [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


Q1: Token ID has no semantic meaning on its own. It is just an ID assigned to a token depending on the order words were added when the vocabulary was built (usually by frequency), which has nothing to do with meaning.

Q2: Words such as learner were split to learn and er. it was split that way because the tokenizer using byte pair encoding recognizes common whole words as one token, while rare or unfamiliar words get split into smaller,still-meaningful pieces. for the word learner it recognized learn as a common word and er as a common suffix.

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentences = [
"The cat sat on the mat.",
"A dog is running in the park.",
"I am learning about embeddings.",
"Rofaida is a computer science learner.",
"Nothing beats a jet2 holiday."
]
embeddings = model.encode(sentences)
print("Shape:", embeddings.shape)
print("First 10 values of sentence 0:", embeddings[0][:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (5, 384)
First 10 values of sentence 0: [ 0.1302372  -0.01577281 -0.03671672  0.05798643 -0.05979176  0.03305371
  0.03012398  0.0289272  -0.01860527  0.05529655]


Q3: Shape: (5, 384)
It depends on the model. This model reads the tokens of each sentence and turns each one to a fixed size 384-number vector no mater how many words are there in each sentence.

Q4:  Individual numbers inside an embedding vector don’t have a hand-assigned meaning, the model learns these numbers as it trains on these words. The meaning is represented by the overall position and direction of the vector in the embedding space. Sentences with similar meanings tend to have embeddings pointing in similar directions, allowing their semantic similarity to be measured.

In [7]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [8]:
sentences = [
    "The cat sat on the mat.",                        # 0
    "A feline rested on the rug.",                     # 1
    "The stock market fell sharply today.",            # 2
    "Investors were alarmed as share prices dropped.", # 3
    "I love sunny weather in the summer.",              # 4
    "The bank is next to the river.",                   # 5
    "The bank approved my loan application.",           # 6
]
embeddings = model.encode(sentences)

In [9]:
pairs = [(0, 1), (2, 3), (0, 4), (5, 6)]
for i, j in pairs:
    score = cosine_similarity(embeddings[i], embeddings[j])
    print(f"{score:.3f}  |  sentence {i}  vs  sentence {j}")

0.556  |  sentence 0  vs  sentence 1
0.554  |  sentence 2  vs  sentence 3
-0.108  |  sentence 0  vs  sentence 4
0.366  |  sentence 5  vs  sentence 6


Q5: Because embeddings capture semantic meanings not just the identical words. cat and feline kind of represent the same concept, sat on and rested on also represent similar meanings. so the cosine similarity was relatively high.

Q6: Same concept here, lexical similarity doesn't mean necessarily mean semantic similarity that the cosine similarity measures based on embedding victors. here each of the words "bank" got assigned by the model to different embedding victor according to the semantic meaning and the cotextual meaning from the surrounding words. the word bank represented a river bank in one sentence and a financial institution in the other.

In [12]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

def word_vector(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    hidden = outputs.last_hidden_state[0]                # (seq_len, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    idx = tokens.index(word)
    return hidden[idx].numpy()

v_river = word_vector("The bank is next to the river.", "bank")
v_loan  = word_vector("The bank approved my loan.", "bank")

print("bank(river) vs bank(loan):", cosine_similarity(v_river, v_loan))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bank(river) vs bank(loan): 0.5222997


In [13]:
import torch
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()

def word_vector(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    hidden = outputs.last_hidden_state[0]                # (seq_len, 768)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    idx = tokens.index(word)
    return hidden[idx].numpy()

v_bank = word_vector("The bank is next to the river.", "bank")
v_river  = word_vector("The bank is next to the river.", "river")

print("bank(river) vs bank(loan):", cosine_similarity(v_bank, v_river))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


bank(river) vs bank(loan): 0.55555105


Q7: Same word/different sentence (bank & bank):0.5222997

Same sentence/different word (bank & river):0.55555105

the cosine similarity score is higher in the second case because it shows more similarity in semantic meaning than in first case.

Q8: It would be 1.00 assigning one fixed victor to the word.

Q9: An example of real problem could be that if I had a financial concern and searched using the word bank on a research engin that uses static embedding it would get mixed up or less relevant results that discusses river banks.

Q10: The actual difference between static and contextual embedding is that in static embedding the model assigns one fixed victor to each word. In real life this forms a huge problem whare there are words that can represent different meanings. Here comes the contextual embedding to solve this problem.
In cotextual embedding the model depends on the surrounding vocabulary of the target word to calculate the context this word is used in, and according to that it assigns a victor that demonstrates that specific meaning of this word. Therefore, a word can hold different meanings without mixing them up.